In [0]:

import random
import pandas as pd
from faker import Faker

fake = Faker()

random.seed(42)
Faker.seed(42)

In [0]:


num_customers = 500
num_products = 500
num_orders = 1000
num_order_items = 2000

In [0]:
customers = []

for i in range(1, num_customers + 1):

    email = fake.email()

    # make some emails invalid
    if random.random() < 0.02:
        email = random.choice([
            "invalidemail.com",
            "user@",
            "abc.com"
        ])

    customers.append({
        "customer_id": f"C{i:04d}",
        "customer_name": fake.name(),
        "email": email,
        "registration_date": fake.date_between(
            start_date="-2y",
            end_date="today"
        ),
        "customer_type": random.choice([
            "REGULAR",
            "PREMIUM",
            "VIP"
        ])
    })

customers_df = pd.DataFrame(customers)

customers_df.head()

,customer_id,customer_name,email,registration_date,customer_type
0,C0001,Brian Yang,johnsonjoshua@example.org,2025-09-09,VIP
1,C0002,Lance Hoffman,fjohnson@example.org,2026-01-12,PREMIUM
2,C0003,Abigail Shaffer,lrobinson@example.com,2025-12-30,REGULAR
3,C0004,Brent Abbott,joshua35@example.org,2025-10-22,VIP
4,C0005,Edward Fuller,blairamanda@example.com,2026-05-16,VIP


In [0]:
categories = {
    "Electronics": ["Laptop", "Phone", "Headphones", "Keyboard"],
    "Clothing": ["T-Shirt", "Jeans", "Jacket", "Shoes"],
    "Home": ["Chair", "Table", "Lamp", "Bedsheet"],
    "Books": ["Novel", "Textbook", "Comics", "Magazine"]
}

products = []

for i in range(1, num_products + 1):

    category = random.choice(list(categories.keys()))
    product_name = random.choice(categories[category])

    # add some messy product names
    if random.random() < 0.10:
        product_name = random.choice([
            " " + product_name,
            product_name + " ",
            product_name.upper(),
            product_name.lower()
        ])

    products.append({
        "product_id": f"P{i:04d}",
        "product_name": product_name,
        "category": category,
        "subcategory": product_name.strip(),
        "cost_price": round(random.uniform(100, 50000), 2)
    })

products_df = pd.DataFrame(products)

products_df.head()

,product_id,product_name,category,subcategory,cost_price
0,P0001,Phone,Electronics,Phone,46054.36
1,P0002,T-Shirt,Clothing,T-Shirt,5012.31
2,P0003,Keyboard,Electronics,Keyboard,15043.36
3,P0004,Laptop,Electronics,Laptop,2900.35
4,P0005,Lamp,Home,Lamp,12285.45


In [0]:
statuses = [
    "PLACED",
    "SHIPPED",
    "DELIVERED",
    "CANCELLED",
    "RETURNED"
]

regions = ["NR", "SR", "ER", "WR"]

orders = []

customer_ids = customers_df["customer_id"].tolist()

for i in range(1, num_orders + 1):

    # around 5% missing customer_id
    if random.random() < 0.05:
        customer_id = None
    else:
        customer_id = random.choice(customer_ids)

    order_date = fake.date_time_between(
        start_date="-1y",
        end_date="now"
    )

    # some dates in wrong format
    if random.random() < 0.05:
        order_date = order_date.strftime("%d-%m-%Y")
    else:
        order_date = order_date.strftime("%Y-%m-%d %H:%M:%S")

    orders.append({
        "order_id": f"O{i:05d}",
        "customer_id": customer_id,
        "order_date": order_date,
        "status": random.choice(statuses),
        "region_code": random.choice(regions)
    })

orders_df = pd.DataFrame(orders)

orders_df.head()

,order_id,customer_id,order_date,status,region_code
0,O00001,C0378,2026-02-21 16:15:56,RETURNED,NR
1,O00002,C0181,2026-04-04 09:35:14,RETURNED,SR
2,O00003,C0097,2025-10-01 20:27:59,CANCELLED,ER
3,O00004,C0020,2025-12-29 22:07:54,SHIPPED,SR
4,O00005,None,2026-06-30 03:42:47,DELIVERED,SR


In [0]:
order_items = []

order_ids = orders_df["order_id"].tolist()
product_ids = products_df["product_id"].tolist()

for i in range(1, num_order_items + 1):

    order_id = random.choice(order_ids)

    # create a few invalid order IDs
    if random.random() < 0.01:
        order_id = "O99999"

    # negative quantity means return
    if random.random() < 0.03:
        quantity = -random.randint(1, 5)
    else:
        quantity = random.randint(1, 5)

    order_items.append({
        "item_id": f"I{i:05d}",
        "order_id": order_id,
        "product_id": random.choice(product_ids),
        "quantity": quantity,
        "unit_price": round(random.uniform(100, 50000), 2),
        "discount_percent": round(random.uniform(0, 100), 2)
    })

order_items_df = pd.DataFrame(order_items)

order_items_df.head()

,item_id,order_id,product_id,quantity,unit_price,discount_percent
0,I00001,O00860,P0300,2,42050.14,28.13
1,I00002,O00166,P0046,3,19825.71,12.30
2,I00003,O00144,P0108,4,30561.22,95.74
3,I00004,O00326,P0386,3,35416.93,27.30
4,I00005,O00679,P0492,4,30015.73,1.59


In [0]:
print("Customers:", len(customers_df))
print("Products:", len(products_df))
print("Orders:", len(orders_df))
print("Order items:", len(order_items_df))

print("\nMissing customer IDs:",
      orders_df["customer_id"].isna().sum())

print("Negative quantities:",
      (order_items_df["quantity"] < 0).sum())

print("Invalid order IDs:",
      (~order_items_df["order_id"].isin(order_ids)).sum())

Customers: 500
Products: 500
Orders: 1000
Order items: 2000

Missing customer IDs: 54
Negative quantities: 60
Invalid order IDs: 13


In [0]:
%sql
SELECT current_catalog(), current_schema();

current_catalog(),current_schema()
workspace,default


In [0]:
base_path = "/Volumes/workspace/default/ecommerce_data/raw"

dbutils.fs.mkdirs(base_path)

print("Raw folder created successfully!")

Raw folder created successfully!


In [0]:
customers_df.to_csv(
    f"{base_path}/customers.csv",
    index=False
)

products_df.to_csv(
    f"{base_path}/products.csv",
    index=False
)

orders_df.to_csv(
    f"{base_path}/orders.csv",
    index=False
)

order_items_df.to_csv(
    f"{base_path}/order_items.csv",
    index=False
)

print("All raw CSV files generated successfully!")

All raw CSV files generated successfully!


In [0]:
display(dbutils.fs.ls(base_path))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/ecommerce_data/raw/customers.csv,customers.csv,30367,1786217296000
dbfs:/Volumes/workspace/default/ecommerce_data/raw/order_items.csv,order_items.csv,73068,1786217297000
dbfs:/Volumes/workspace/default/ecommerce_data/raw/orders.csv,orders.csv,44208,1786217297000
dbfs:/Volumes/workspace/default/ecommerce_data/raw/products.csv,products.csv,18694,1786217296000
